# Reed-Solomon Decoder

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport classes.galois_field
%aimport classes.galois_poly

from classes.galois_field import *
from classes.galois_poly import *
from classes.rs_decoder import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [126]:
def do_unpack(value, degree, gf):
    mask    = gf._order
    ak      = [(value >> i*gf._field_m) & mask for i in range(degree+1)]
    ak      = [gf._get_power(a) for a in ak]
    return GaloisPoly(gf, ak[::-1])

## Get Syndromes

In [127]:
gf_extended = GaloisField(4, 0b10011)
root        = [GFE1, 1, 2, 3, 4, 5]

r   = 0x56D09_F171B_1E06F
r   = do_unpack(r, 14, gf_extended)

syn = [r(a) for a in root]

syn


Field Closed Succesfully!, 15 Non-Zero Elements


[1, 14, 0, 10, 9, 13]

In [128]:
def get_discrepancy(mu, syn, Lx):
    d_mu    = syn[mu]
    for i in range(1, Lx._degree+1): 
        # passes through all coeffs of localizator polynomial 
        term    = gf_extended.mul(Lx @ i, syn[mu - i])[ID_POWER]
        # Lx @ i take coefficient of degree "i" from the locator polynomial
        # syn[mu - i]
        d_mu    = gf_extended.add(d_mu, term)[ID_POWER]
        # discrpancy in step i
    return d_mu

In [129]:
# berlekamp-massey by steps
x   = do_unpack(0x10, 1, gf_extended)

lm1 = do_unpack(0x1, 0, gf_extended)
dm1 = GFE1

l0  = do_unpack(0x1, 0, gf_extended)
d0  = syn[0]

dx  = x*gf_extended.div(d0, dm1)[ID_POWER]
l1  = l0 + dx*lm1
d1  = get_discrepancy(1, syn, l1)

dx, l1, d1
print(l1, "\n")

dx  = x*gf_extended.div(d1, d0)[ID_POWER]
l2  = l1 + dx*l0
d2  = get_discrepancy(2, syn, l2)

dx, l2, d2
print(l2, "\n")

dx  = (x*x)*gf_extended.div(d2, d0)[ID_POWER]
l3  = l2 + dx*l0
d3  = get_discrepancy(3, syn, l3)

dx, l3, d3
print(l3, "\n")

l4  = l3
d4  = get_discrepancy(4, syn, l4)

l4, d4
print(l4, "\n")

dx  = (x*x)*gf_extended.div(d4, d2)[ID_POWER]
l5  = l4 + dx*l2
d5  = get_discrepancy(5, syn, l5)

dx, l5, d5
print(l5, "\n")

dx  = (x)*gf_extended.div(d5, d4)[ID_POWER]
l6  = l5 + dx*l4

dx, l6
print(l6, "\n")


a1*x + 1 

a13*x + 1 

a10*x2 + a13*x + 1 

a10*x2 + a13*x + 1 

a5*x3 + a6*x2 + a13*x + 1 

a4*x3 + a1*x2 + 1 



In [130]:
print(do_unpack(0x10, 1, gf_extended))

x


## Chiens evaluation

In [131]:
for k in range(l6._degree + 1):
    print(k, l6 @ k)
    

0 0
1 -1
2 1
3 4


In [132]:
# (ascending power: a0, a1, ..., an)
for k in range(l6._degree + 1):
    alpha_k_idx = l6 @ k                             # alpha power of coeff of x^k (GFE0 = -1 -> zero)
    value, pw   = gf_extended(alpha_k_idx)           # (binary value, alpha power)
    alpha       = "0" if alpha_k_idx == GFE0 else f"a^{pw}"
    print(f"x^{k}: {alpha:5} = {value:0{gf_extended._field_m}b}")

x^0: a^0   = 0001
x^1: 0     = 0000
x^2: a^1   = 0010
x^3: a^4   = 0011


In [133]:
# Chien search: error at position i if L(alpha^-i) == 0
#   alpha^(-i) = alpha^(15-i)  ->  gf.inv(i)[ID_POWER]
def _chien_search(Lx, gf):
    err_pos = []
    for i in range(gf._order):
        a_inv = gf.inv(i)[ID_POWER]         # alpha^-i (power index)
        if Lx(a_inv) == GFE0:               # root of the locator polynomial
            err_pos.append(i)
    if len(err_pos) != Lx._degree:
        print(f"Warning: {len(err_pos)} roots found, expected {Lx._degree} (uncorrectable)")
    return err_pos

err_pos = _chien_search(l6, gf_extended)
err_pos

[0, 6, 13]

In [ ]:
gf256   = GaloisField(8, 0b1_0001_1101)
N, K    = 44, 28
N_PAR   = N - K                 # 2t = 16 roots
B0      = 0                     # first root alpha^0

# generador y codificador sistematico 
g       = GaloisPoly(gf256, [GFE1]).poly_from_roots(list(range(B0, B0 + N_PAR)))
x_16    = GaloisPoly(gf256, [GFE1] + [GFE0]*N_PAR)

# g(x): generator
# x_16: monomial x^16

def rs_encode(msg_bytes):
    """K bytes -> N bytes (msg + parity)"""
    m   = GaloisPoly(gf256, [gf256._get_power(b) for b in msg_bytes])
    r   = (m * x_16) % g
    # Compute parity
    # gf256(e)[ID_BIN] takes the first element of the tuple
    par = [gf256(e)[ID_BIN] for e in r._coeff]
    # 28 msg bytes + remaining zeros + parity
    return list(msg_bytes) + [0]*(N_PAR - len(par)) + par

#g
#x_16

Field Closed Succesfully!, 255 Non-Zero Elements


## Call-Backs Decoder

In [ ]:
def coeff(p, i):
    """Cofficient (alpha power) of x^i, GFE0 if isn't exists"""
    # p @ i, return the coeff of x^i
    return GFE0 if (p._zero_poly or i > p._degree) else p @ i 

def bytes_to_poly(gf, data):
    """Convert a list of bytes into a GaloisPoly object"""
    return GaloisPoly(gf, [gf._get_power(b) for b in data])

def idx_to_loc(j, n=N):
    """Byte index -> locator power X = alpha^(n-1-j)
    
    i.e. 4 bytes

    rx = [0xA, 0xB, 0xC, 0xD]
          j=0  j=1  j=2  j=3

    convert to:

    r(x) = A·x³ + B·x² + C·x¹ + D·x⁰
    
    """
    return n - 1 - j


def get_syndromes_rs(gf, r_poly, n_parity=N_PAR, b0=B0):
    # Valuate poly with roots (16)
    return [r_poly(b0 + i) for i in range(n_parity)] # n_parity = 16

def get_erasure_locator(gf, eras_idx, n=N):
    """Gamma(x) = prod (1 + Y_l x)"""
    Gx = GaloisPoly(gf, [GFE1])
    for j in eras_idx:
        Gx = Gx * GaloisPoly(gf, [idx_to_loc(j, n), GFE1])     # Y_l*x + 1
    return Gx

def berlekamp_massey_eras(gf, syn, Gx, n_eras):
    """ BM initialized with erasured locator. return (Psi, L)"""
    x   = GaloisPoly(gf, [GFE1, GFE0])
    Lx  = Gx
    Bx  = Gx
    L   = n_eras
    for r in range(n_eras, len(syn)):
        # discrepancia
        delta = GFE0
        for i in range(min(Lx._degree, r) + 1):
            delta = gf.add(delta, gf.mul(coeff(Lx, i), syn[r - i])[ID_POWER])[ID_POWER]

        Bx = Bx * x
        if delta != GFE0:
            Tx = Lx + (Bx * delta)
            if 2*L <= r + n_eras:
                Bx = Lx * gf.inv(delta)[ID_POWER]
                L  = r + 1 + n_eras - L
            Lx = Tx
    return Lx, L

def chien_search_rs(gf, Px, n=N):
    """Recorre solo las n posiciones existentes. Devuelve indices de byte j con Px(X^-1) == 0"""
    pos = []
    for j in range(n):
        x_inv = gf.inv(idx_to_loc(j, n))[ID_POWER]
        if Px(x_inv) == GFE0:
            pos.append(j)
    return pos

def forney_rs(gf, syn, Px, pos, n=N, b0=B0):
    """Valores de error/borrado en las posiciones encontradas"""
    Sx  = GaloisPoly(gf, syn[::-1])
    Ox  = (Sx * Px) % GaloisPoly(gf, [GFE1] + [GFE0]*len(syn))     # Omega = S*Psi mod x^2t
    Pxp = Px.formal_derivative()
    values = []
    for j in pos:
        X     = idx_to_loc(j, n)
        x_inv = gf.inv(X)[ID_POWER]
        den   = Pxp(x_inv)
        if den == GFE0:
            return None                                             # raiz multiple
        num   = gf.mul(Ox(x_inv), gf.pow(X, 1 - b0)[ID_POWER])[ID_POWER]
        values.append(gf.div(num, den)[ID_POWER])
    return values

In [136]:
def rs_decode(rx, erasures=(), gf=gf256, n=N, k=K, verbose=False):
    """
    Decodificador RS con correccion conjunta de errores y borrados.

    Args:
        rx:       N bytes recibidos (int)
        erasures: indices de byte borrados (0 = primer byte)

    Returns:
        tuple: (mensaje [K bytes], patron_error [N bytes], corregido, no_corregible)
    """
    n_par   = n - k
    rx      = list(rx)
    eras    = sorted(set(erasures))
    f       = len(eras)
    fail    = (rx[:k], [0]*n, False, True)

    if f > n_par or any(j < 0 or j >= n for j in eras):
        return fail

    # 1. sindromes
    syn = get_syndromes_rs(gf, bytes_to_poly(gf, rx), n_par)
    if all(s == GFE0 for s in syn):
        return rx[:k], [0]*n, False, False

    # 2-3. localizador de errores y borrados
    Gx      = get_erasure_locator(gf, eras, n)
    Px, L   = berlekamp_massey_eras(gf, syn, Gx, f)
    nu      = L - f
    if verbose:
        print(f"f={f}, nu={nu}, L={L}, deg(Psi)={Px._degree}")
    if Px._degree != L or 2*nu + f > n_par:
        return fail

    # 4. Chien sobre las n posiciones existentes
    pos = chien_search_rs(gf, Px, n)
    if verbose:
        print("posiciones:", pos)
    if len(pos) != Px._degree:
        return fail                                     # alguna raiz fuera de las n posiciones

    # 5. Forney
    values = forney_rs(gf, syn, Px, pos, n)
    if values is None:
        return fail

    err = [0]*n
    for j, e in zip(pos, values):
        if e == GFE0 and j not in eras:
            return fail                                 # error de valor nulo: inconsistente
        err[j] = gf(e)[ID_BIN]

    corr = [a ^ e for a, e in zip(rx, err)]

    # 6. verificacion
    if any(s != GFE0 for s in get_syndromes_rs(gf, bytes_to_poly(gf, corr), n_par)):
        return fail

    return corr[:k], err, any(err), False

In [137]:
import random
random.seed(1)

msg = list(range(K))
cw  = rs_encode(msg)
print(' '.join(f'{b:02X}' for b in cw))

def corrupt(cw, err_idx, eras_idx):
    rx = cw.copy()
    for j in err_idx:
        rx[j] ^= random.randint(1, 255)         # error (valor no nulo)
    for j in eras_idx:
        rx[j] = 0                               # borrado: se reemplaza por 0
    return rx

def run_case(name, n_err, n_eras):
    idx      = random.sample(range(N), n_err + n_eras)
    err_idx  = idx[:n_err]
    eras_idx = idx[n_err:]
    rx       = corrupt(cw, err_idx, eras_idx)
    m, e, corrected, uncorrectable = rs_decode(rx, eras_idx)
    ok       = (m == msg) and not uncorrectable
    print(f"{name:28} 2v+f={2*n_err+n_eras:2}  corregido={corrected!s:5}  no_corregible={uncorrectable!s:5}  mensaje_ok={ok}")
    return m, e, corrected, uncorrectable

run_case("sin errores",                 0,  0)
run_case("8 errores",                   8,  0)
run_case("16 borrados",                 0, 16)
run_case("4 errores + 8 borrados",      4,  8)
run_case("6 errores + 4 borrados",      6,  4)
run_case("1 error + 14 borrados",       1, 14)
run_case("9 errores (excede)",          9,  0)
run_case("5 errores + 8 borrados (exc)",5,  8)
run_case("17 borrados (excede)",        0, 17)

00 01 02 03 04 05 06 07 08 09 0A 0B 0C 0D 0E 0F 10 11 12 13 14 15 16 17 18 19 1A 1B E8 9B BB 6A 6F 00 3E 3E C9 7B C7 1E 8E 75 65 38
sin errores                  2v+f= 0  corregido=False  no_corregible=False  mensaje_ok=True
8 errores                    2v+f=16  corregido=True   no_corregible=False  mensaje_ok=True
16 borrados                  2v+f=16  corregido=True   no_corregible=False  mensaje_ok=True
4 errores + 8 borrados       2v+f=16  corregido=True   no_corregible=False  mensaje_ok=True
6 errores + 4 borrados       2v+f=16  corregido=True   no_corregible=False  mensaje_ok=True
1 error + 14 borrados        2v+f=16  corregido=True   no_corregible=False  mensaje_ok=True
9 errores (excede)           2v+f=18  corregido=False  no_corregible=True   mensaje_ok=False
5 errores + 8 borrados (exc) 2v+f=18  corregido=False  no_corregible=True   mensaje_ok=False
17 borrados (excede)         2v+f=17  corregido=False  no_corregible=True   mensaje_ok=False


([0,
  1,
  2,
  0,
  4,
  5,
  6,
  7,
  0,
  9,
  10,
  0,
  12,
  0,
  14,
  15,
  16,
  0,
  0,
  19,
  20,
  21,
  22,
  23,
  0,
  25,
  26,
  0],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 False,
 True)

In [138]:
# detalle de un caso: patron de error devuelto vs. el introducido
err_idx  = [2, 10, 30]
eras_idx = [5, 20, 40, 43]
rx       = corrupt(cw, err_idx, eras_idx)
m, e, corrected, uncorrectable = rs_decode(rx, eras_idx, verbose=True)

print("rx :", ' '.join(f'{b:02X}' for b in rx))
print("err:", ' '.join(f'{b:02X}' for b in e))
print("msg:", ' '.join(f'{b:02X}' for b in m))
print("patron == rx ^ cw:", e == [a ^ b for a, b in zip(rx, cw)])
print("corregido:", corrected, " no_corregible:", uncorrectable)

f=4, nu=3, L=7, deg(Psi)=7
posiciones: [2, 5, 10, 20, 30, 40, 43]
rx : 00 01 36 03 04 00 06 07 08 09 FB 0B 0C 0D 0E 0F 10 11 12 13 00 15 16 17 18 19 1A 1B E8 9B 39 6A 6F 00 3E 3E C9 7B C7 1E 00 75 65 00
err: 00 00 34 00 00 05 00 00 00 00 F1 00 00 00 00 00 00 00 00 00 14 00 00 00 00 00 00 00 00 00 82 00 00 00 00 00 00 00 00 00 8E 00 00 38
msg: 00 01 02 03 04 05 06 07 08 09 0A 0B 0C 0D 0E 0F 10 11 12 13 14 15 16 17 18 19 1A 1B
patron == rx ^ cw: True
corregido: True  no_corregible: False


In [139]:
# prueba aleatoria: todo patron con 2v+f <= 16 debe corregirse
fails = 0
for _ in range(200):
    msg_r = [random.randint(0, 255) for _ in range(K)]
    cw_r  = rs_encode(msg_r)
    f     = random.randint(0, N_PAR)
    v     = random.randint(0, (N_PAR - f) // 2)
    idx   = random.sample(range(N), v + f)
    rx    = corrupt(cw_r, idx[:v], idx[v:])
    m, e, c, u = rs_decode(rx, idx[v:])
    if u or m != msg_r or e != [a ^ b for a, b in zip(rx, cw_r)]:
        fails += 1
print("fallos:", fails, "de 200")

fallos: 0 de 200
